In [10]:
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics.pairwise import cosine_similarity

In [11]:
df = pd.read_csv(
    "../data/processed/cases.csv"
)

In [12]:
vectorizer = joblib.load(
    "../data/processed/tfidf_vectorizer.pkl"
)

In [13]:
docs = df["ringkasan_fakta"].fillna("").astype(str)
X = vectorizer.transform(docs)
print("Nulls in ringkasan_fakta:", df["ringkasan_fakta"].isna().sum())


Nulls in ringkasan_fakta: 5


In [14]:
def retrieve_cases(query, top_k=5):

    query_vec = vectorizer.transform([query])

    similarities = cosine_similarity(
        query_vec,
        X
    )[0]

    top_indices = np.argsort(
        similarities
    )[-top_k:][::-1]

    results = df.iloc[top_indices].copy()

    results["similarity"] = similarities[top_indices]

    return results

In [15]:
from collections import Counter

def predict_decision(query):

    retrieved = retrieve_cases(
        query,
        top_k=5
    )

    labels = retrieved["label"]

    prediction = Counter(
        labels
    ).most_common(1)[0][0]

    return prediction

In [18]:
def explain_prediction(query):

    retrieved = retrieve_cases(
        query,
        top_k=5
    )

    prediction = (
        retrieved["label"]
        .value_counts()
        .idxmax()
    )

    print("Prediksi Putusan:")
    print(prediction)

    print("\nKasus Pendukung:")

    display(
        retrieved[
            [
                "nomor_perkara",
                "label",
                "similarity"
            ]
        ]
    )

In [21]:
query = """
Penggugat membeli tanah
namun sertifikat belum dibalik nama.
"""

predict_decision(query)
explain_prediction(query)

Prediksi Putusan:
Dikabulkan Sebagian

Kasus Pendukung:


,nomor_perkara,label,similarity
7,192/Pdt.G/2020/PN,Dikabulkan Sebagian,0.176284
15,227/Pdt.G/2020/PN,Dikabulkan Sebagian,0.174791
12,207/Pdt.G/2020/PN,Dikabulkan Sebagian,0.174602
6,185/Pdt.G/2020/PN,Dikabulkan Sebagian,0.174157
9,197/Pdt.G/2020/PN,Dikabulkan Sebagian,0.174095
